## Install Dependency

In [1]:
import os
from dotenv import load_dotenv
from typing import Any, Dict, List
from pydantic import PrivateAttr
import pandas as pd
import numpy as np

from distilabel.llms import OpenAILLM
from distilabel.pipeline import Pipeline
from distilabel.steps import (
    LoadDataFromDicts,
    MergeColumns,
    FormatTextGenerationDPO,
    PreferenceToArgilla
)
from distilabel.steps.tasks import TextGeneration, UltraFeedback
from distilabel.typing import ChatType


/tmp/ipykernel_116548/2111529966.py:8: DeprecationWarning: Importing from 'distilabel.llms' is deprecated and will be removed in a version 1.7.0. Import from 'distilabel.models' instead.
  from distilabel.llms import OpenAILLM
/data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'default' a

In [2]:
load_dotenv()

api_key = os.getenv('UPSTAGE_API_KEY')

## Define the pipeline

In [3]:
df_aihub = pd.read_json('/data/ephemeral/pro-nlp-finalproject-nlp-13/data/persona_data/AiHub_100sample.json', lines=True)

df_aihub.rename(columns={"req": "instruction"}, inplace=True)

data_dicts = df_aihub.to_dict(orient="records")

In [4]:
SNS_CONVERSATION_CRITERIA = {
    "naturalness": {
        "description": "대화가 자연스럽고 실제 SNS에서 사용되는 표현과 어투를 사용하는가",
        "weight": 0.25,
        "rubric": """
        5: 매우 자연스러운 SNS 대화체, 이모티콘/줄임말 적절히 사용
        4: 자연스러운 대화체이나 약간의 어색함
        3: 보통 수준, SNS 특성이 일부 반영됨
        2: 딱딱하거나 형식적인 표현 다수
        1: SNS 대화체와 거리가 먼 격식체
        """
    },
    "contextual_relevance": {
        "description": "이전 대화 맥락을 잘 이해하고 적절하게 응답하는가",
        "weight": 0.25,
        "rubric": """
        5: 맥락을 완벽히 이해하고 자연스럽게 이어감
        4: 맥락을 잘 이해하나 일부 어색한 부분
        3: 맥락을 대체로 이해하나 연결이 약함
        2: 맥락 이해가 부족하거나 동떨어진 응답
        1: 맥락을 완전히 무시한 응답
        """
    },
    "engagement": {
        "description": "대화 상대의 참여를 유도하고 흥미를 유지하는가",
        "weight": 0.20,
        "rubric": """
        5: 공감, 질문, 유머 등으로 활발한 대화 유도
        4: 대화를 이어가기 좋은 응답
        3: 무난하지만 특별히 흥미롭지 않음
        2: 대화를 끊을 수 있는 단답형 응답
        1: 대화 참여 의지가 전혀 느껴지지 않음
        """
    },
    "emotional_appropriateness": {
        "description": "상황과 감정에 적절한 톤과 공감을 표현하는가",
        "weight": 0.15,
        "rubric": """
        5: 상황에 완벽히 맞는 감정 표현과 공감
        4: 적절한 감정 표현
        3: 보통 수준의 감정 반영
        2: 부적절하거나 어긋난 감정 표현
        1: 감정 무시 또는 완전히 부적절한 반응
        """
    },
    "conciseness": {
        "description": "SNS 특성에 맞게 간결하면서도 충분한 정보를 전달하는가",
        "weight": 0.15,
        "rubric": """
        5: 간결하면서 핵심을 정확히 전달
        4: 적절한 길이로 정보 전달
        3: 약간 장황하거나 짧지만 이해 가능
        2: 너무 길거나 짧아서 의미 전달 부족
        1: 극단적으로 부적절한 길이
        """
    }
}

In [5]:
class SNSConversationFeedback(UltraFeedback):
    """SNS 대화체에 특화된 평가 태스크"""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.aspect = "overall-rating"

    _system_prompt: str = PrivateAttr(
        default="""당신은 실제 SNS 대화체(카카오톡, DM 등)를 판별하는 전문 평가가입니다.

당신의 출력은 오직 아래에 정의된 출력 문법만 허용됩니다.
이를 벗어나는 모든 출력은 파싱 오류로 처리됩니다.

[출력 제약 – 절대 위반 금지]
- 출력은 정확히 4줄이어야 합니다.
- 각 응답은 다음 두 줄로만 구성됩니다:
  1) Rating: 숫자
  2) Rationale: 문장
- 두 응답 묶음(Rating, Rationale 쌍) 사이에는 정확히 빈 줄 한 줄만 허용됩니다. 
- 마지막 Rationale 줄 뒤에는 줄바꿈이나 공백 없이 즉시 종료해야 합니다. 
- Rating에는 오직 1~5 사이의 실수만 허용됩니다.
- Rationale은 반드시 한 문장이어야 합니다.
- Rating, Rationale 외의 어떤 텍스트도 출력해서는 안 됩니다.

[금지 사항]
- 추가 설명, 요약, 평가 근거, 분석
- <think> 태그 또는 내부 사고 노출
- 구분선(---), 제목, 번호, 마크다운 기호
- 출력 전후의 공백 줄 또는 문장

[출력 문법]
Rating: [1-5]
Rationale: [한 문장]

Rating: [1-5]
Rationale: [한 문장]
"""
    )

    def format_input(self, input: Dict[str, Any]) -> ChatType:
        """SNS 대화체 판별 및 품질 평가 전용 프롬프트"""

        system_content = self._system_prompt

        # 1. 평가 목적
        user_content = (
            "아래 응답들이 실제 SNS 대화체인지 평가하십시오.\n"
            "평가는 자연스러움, 맥락 이해, 참여 유도, 감정 적절성, 간결성을 종합하여 수행하십시오.\n\n"
        )

        # 2. 다차원 평가 기준
        user_content += (
            "[평가 기준]\n"
            "- 자연스러움: 구어체, 이모티콘, 줄임말 사용 여부\n"
            "- 맥락 적합성: 이전 대화를 이해하고 이어지는가\n"
            "- 참여 유도: 공감, 질문 등 대화 지속 요소\n"
            "- 감정 적절성: 상황에 맞는 톤과 공감\n"
            "- 간결성: SNS 특성에 맞는 길이\n\n"
            "주의: 설명조, 보고서체, 비즈니스 이메일 말투는 낮은 점수를 부여하십시오.\n\n"
        )

        # 3. 대화 맥락
        user_content += f"[지시 사항]\n{input['instruction']}\n\n"

        # 4. 응답 나열
        for i, gen in enumerate(input["generations"], 1):
            user_content += f"[응답 {i}]\n{gen}\n\n"

        # 5. 형식 리마인드 (최소화)
        user_content += "출력은 system에 정의된 출력 문법만 따르십시오."

        return [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
        ]


In [6]:
from distilabel.steps.base import Step, StepInput
from typing import Iterable

class FilterBadRatings(Step):
    @property
    def inputs(self) -> list[str]:
        return ["instruction", "generations", "ratings", "rationales"]

    @property
    def outputs(self) -> list[str]:
        return ["instruction", "generations", "ratings", "rationales"]

    def process(self, inputs: StepInput) -> Iterable[list[dict]]:
        filtered_batch = []
        target_columns = [
            'instruction', 'distilabel_metadata', 'model_name', 
            'generations', 'ratings', 'rationales'
        ]
        
        for item in inputs:
            ratings = item.get("ratings")
            if (isinstance(ratings, (list, np.ndarray)) and 
                len(ratings) == 2 and 
                not pd.isna(ratings).any()):
                
                # 에러를 유발하는 'chosen_model', 'rejected_model' 등을 제외하고 복사
                clean_item = {k: v for k, v in item.items() if k in target_columns}
                filtered_batch.append(clean_item)
        
        yield filtered_batch

In [7]:
with Pipeline(name="sns-conversation-dpo-pipeline") as pipeline:
        load_data = LoadDataFromDicts(
            data=data_dicts[:10],
            batch_size=5
        )

        generate_responses = [
            TextGeneration(
                llm=OpenAILLM(
                    model="solar-mini",
                    base_url="https://api.upstage.ai/v1/solar",
                    api_key="up_KyCFVhEHCYkyBTmQOxujiKk55YVAL",
                    max_retries=10,
                ),
                system_prompt="""당신은 문서체로만 텍스트를 생성해야합니다. 주 목표는 SNS 대화체와 정반대의 성질을 가지는 말투를 가져야 합니다.""",
            )
        ]

        combine_responses = MergeColumns(
            columns=["res", "generation"],
            output_column="generations"
        )
        
        evaluate_responses = SNSConversationFeedback(
            aspect="overall-rating",
            llm=OpenAILLM(
                model="solar-pro2",
                base_url="https://api.upstage.ai/v1/solar",
                api_key=api_key,
                max_retries=10,
            ),
        )

        filter_dpo = FilterBadRatings()
        
        format_dpo = FormatTextGenerationDPO()

        load_data >> generate_responses >> combine_responses >> evaluate_responses >> filter_dpo >> format_dpo



In [8]:
distiset = pipeline.run(use_cache=False)

[02/03/26 13:03:43] INFO     ['distilabel.pipeline'] 📝 Pipeline data will be written to               ]8;id=35317;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=317242;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1015\1015]8;;\
                             '/data/ephemeral/home/.cache/distilabel/pipelines/sns-conversation-dpo-pi             
                             peline/abf0ba4f6e0881cf0d4c6e334c1f04ebda265f9a/executions/10ac816f0682e3             
                             30c3b72ffaa3303907143a1bde/data/steps_outputs'                                        

                    INFO     ['distilabel.pipeline'] ⌛ The steps of the pipeline will be loaded in    ]8;id=926363;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=791695;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1046\1046]8;;\
                             stages:                                                                               
                              * Legend: 🚰 GeneratorStep 🌐 GlobalStep 🔄 Step                                     
                              * Stage 0:                                                                           
                                - 🚰 'load_data_from_dicts_0'                                                      
                                - 🔄 'text_generation_0'                                                           
                                - 🔄 'merge_columns_0'                                                             
                                - 🔄 's_n_s_conversation_feedback_0'                                               
                                - 🔄 'filter_bad_ratings_0'                                                        
                                - 🔄 'format_text_generation_d_p_o_0'                                              

                    INFO     ['distilabel.pipeline'] ⏳ Waiting for all the steps of stage 0 to        ]8;id=161138;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=903491;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1382\1382]8;;\
                             load...                                                                               

[02/03/26 13:03:45] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 6/6                 ]8;id=50342;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=91845;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_dicts_0' replicas: 1/1                                             
                              * 'text_generation_0' replicas: 1/1                                                  
                              * 'merge_columns_0' replicas: 1/1                                                    
                              * 's_n_s_conversation_feedback_0' replicas: 1/1                                      
                              * 'filter_bad_ratings_0' replicas: 1/1                                               
                              * 'format_text_generation_d_p_o_0' replicas: 1/1                                     

                    INFO     ['distilabel.pipeline'] ✅ All the steps from stage 0 have been loaded!   ]8;id=948617;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=710101;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1422\1422]8;;\

...

                    INFO     ['distilabel.step.filter_bad_ratings_0'] 🏁 Finished running step  ]8;id=409479;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=221123;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#129\129]8;;\
                             'filter_bad_ratings_0' (replica ID: 0)                                                

                    INFO     ['distilabel.step.format_text_generation_d_p_o_0'] 📦 Processing   ]8;id=887625;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=512543;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#230\230]8;;\
                             batch 0 in 'format_text_generation_d_p_o_0' (replica ID: 0)                           

                    INFO     ['distilabel.step.format_text_generation_d_p_o_0'] 📨 Step         ]8;id=435516;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=901240;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#290\290]8;;\
                             'format_text_generation_d_p_o_0' sending batch 0 to output queue                      

                    INFO     ['distilabel.step.format_text_generation_d_p_o_0'] 🏁 Finished     ]8;id=285711;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=41847;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#129\129]8;;\
                             running step 'format_text_generation_d_p_o_0' (replica ID: 0)                         

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:
df = distiset["default"]["train"].to_pandas()

In [15]:
df.loc[0, 'chosen']

array([{'content': '여름 휴가철 어디로 갈까?', 'role': 'user'},
       {'content': '나는 해변으로 가고 싶어!', 'role': 'assistant'}], dtype=object)

In [30]:
nan_ratings_df = df[df['ratings'].apply(
    lambda x: (not isinstance(x, (list, np.ndarray))) or  # 리스트 형태가 아님
              (len(x) != 2) or                           # 길이가 2가 아님 (보내주신 데이터 중 3개, 6개인 경우)
              (pd.isna(x).any())                         # 내부에 null/NaN이 포함됨
)]

# 2. 결과 확인
print(f"비정상 데이터 개수: {len(nan_ratings_df)}건")
display(nan_ratings_df)

비정상 데이터 개수: 0건


,instruction,distilabel_metadata,model_name,generations,ratings,rationales,prompt,prompt_id,chosen,chosen_rating,rejected,rejected_rating


In [31]:
df.head(10)

,instruction,distilabel_metadata,model_name,generations,ratings,rationales,prompt,prompt_id,chosen,chosen_rating,rejected,rejected_rating
0,여름 휴가철 어디로 갈까?,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[나는 해변으로 가고 싶어!, 여름 휴가 시즌이 다가오고 있습니다. 이번 여름 휴가...","[4, 1]","[자연스럽지만 맥락 이해가 부족하고 간결함 없이 설명을 이어감., 보고서체로 지나치...",여름 휴가철 어디로 갈까?,a5fcfdcddb9fa9c0b8673b038c4e3f0ddd0ec6c3f418ed...,"[{'content': '여름 휴가철 어디로 갈까?', 'role': 'user'}...",4,"[{'content': '여름 휴가철 어디로 갈까?', 'role': 'user'}...",1
1,경기도 청소년 관악제의 목적은 뭐임?,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,[경기도 청소년 관악제는 청소년들에게 예술적 재능을 발휘할 수 있는 기회를 제공하고...,"[3, 1]","[구어체 부재와 설명조 서술로 자연스러움이 떨어지지만, 목적은 명확히 전달함., 지...",경기도 청소년 관악제의 목적은 뭐임?,3e1ccbbb29e4ff6e7186c67bca806418aedd2cf8d30b4d...,"[{'content': '경기도 청소년 관악제의 목적은 뭐임?', 'role': '...",3,"[{'content': '경기도 청소년 관악제의 목적은 뭐임?', 'role': '...",1
2,거창한 책보다는 가벼운 소설이나 만화책을 읽어보는 건 ㅇㄸ?,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[그게 좋을 것 같다. 그래서 쉬는 시간에 책을 읽어야지, 가벼운 소설이나 만화책을...","[3, 1]","[구어체와 간결성이 있으나 참여 유도나 감정 표현이 부족하다., 설명조 톤과 보고서...",거창한 책보다는 가벼운 소설이나 만화책을 읽어보는 건 ㅇㄸ?,474a941bbab41ce4bae63629e45b794e111e65d457b0fa...,[{'content': '거창한 책보다는 가벼운 소설이나 만화책을 읽어보는 건 ㅇㄸ...,3,[{'content': '거창한 책보다는 가벼운 소설이나 만화책을 읽어보는 건 ㅇㄸ...,1
3,렬루? 그럼 어떤 건축양식이 있는 거야?,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[고딕, 르네상스, 바로크 등이 있어! 그리고 1920년에는 합스부르크 왕가가 여름...","[3, 1]",[구어체는 사용되었으나 맥락 이해가 부족하고 감정 표현이 미흡하며 설명이 지나치게 ...,렬루? 그럼 어떤 건축양식이 있는 거야?,31e7c80870025112bc323cead23aa6e61535e4a45b2bfb...,"[{'content': '렬루? 그럼 어떤 건축양식이 있는 거야?', 'role':...",3,"[{'content': '렬루? 그럼 어떤 건축양식이 있는 거야?', 'role':...",1
4,"공항 면세점 상품 구성도 새로 하고, 할인 이벤트도 준비하고 있다고 하더라고.",{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[그렇구나. 잘 준비해서 좋은 결과 있었으면 해., 공항 면세점에 대한 새로운 상품...","[4, 1]","[간결하고 공감을 표현하며 자연스럽게 대화를 이어감., 설명조 문장으로 SNS 대화...","공항 면세점 상품 구성도 새로 하고, 할인 이벤트도 준비하고 있다고 하더라고.",59a72f5da6132e207abd929b22d796e4a281008b4e6c05...,"[{'content': '공항 면세점 상품 구성도 새로 하고, 할인 이벤트도 준비하...",4,"[{'content': '공항 면세점 상품 구성도 새로 하고, 할인 이벤트도 준비하...",1
5,레알로? 그럼 시간 맞춰서 가야겠다!,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[월요일은 쉬니까 주의해!, 실제로 그러시다면, 적절한 시간에 방문하시는 것이 좋겠...","[4, 2]","[구어체와 줄임말 사용이 자연스러우며 참여 유도 요소가 포함됐다., 설명조 어조와 ...",레알로? 그럼 시간 맞춰서 가야겠다!,b328cbb56dd3394cb622010a1c50a02c56b94224bfd88e...,"[{'content': '레알로? 그럼 시간 맞춰서 가야겠다!', 'role': '...",4,"[{'content': '레알로? 그럼 시간 맞춰서 가야겠다!', 'role': '...",2
6,백화점은 삶의 스타일을 팔고 있어,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[그래서 가구도 스타일에 맞춰서 배치하는 거지?, 백화점은 단순한 상품 판매를 넘어...","[4, 1]","[구어체 질문과 맥락 연계로 자연스러운 대화체를 보여주며 참여를 유도한다., 설명조...",백화점은 삶의 스타일을 팔고 있어,b096b0946aa8dff1511ddea83eea90d23653781f8544d7...,"[{'content': '백화점은 삶의 스타일을 팔고 있어', 'role': 'us...",4,"[{'content': '백화점은 삶의 스타일을 팔고 있어', 'role': 'us...",1
7,지리적으로 유명한 곳들을 여행하면 더욱 재미있을 것 같아!,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[그래. 여행 계획을 세워보자!, 지리적 명소에 대한 여행은 교육적 가치가 높으며,...","[4, 1]","[응답 1은 구어체와 간결한 표현으로 자연스럽게 대화를 이끌며, 참여 유도 효과가 ...",지리적으로 유명한 곳들을 여행하면 더욱 재미있을 것 같아!,950a7fbff2e0d8fec664db74303cedabcb49601f728b3e...,[{'content': '지리적으로 유명한 곳들을 여행하면 더욱 재미있을 것 같아!...,4,[{'content': '지리적으로 유명한 곳들을 여행하면 더욱 재미있을 것 같아!...,1
8,가수들의 공연도 있는 거지?,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[맞아! 규현, 김나영, 폴킴 등이 대표적인 드라마 삽입곡으로 공연을 선사해!, 콘...","[4, 2]","[구어체와 줄임말(""사해"")을 사용하며 참여 유도 요소가 있고 간결합니다., 설명조...",가수들의 공연도 있는 거지?,36d7275ec237a4e56512c201fda7ae7bc29d9c33ee58d8...,"[{'content': '가수들의 공연도 있는 거지?', 'role': 'user'...",4,"[{'content': '가수들의 공연도 있는 거지?', 'role': 'user'...",2
9,최근에 뭘 하고 지내?,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[그냥 평범하게 지내고 있어 ㅎㅎ, 최근 들어 여러 방면에서 시간을 보내고 있습니다...","[4, 1]","[구어체와 이모티콘 사용으로 자연스럽고 간결하며 대화 참여 유도 가능, 공식적인 보...",최근에 뭘 하고 지내?,70192399138575b3cb9d0a702cd8ff8b11e077a78c6ce9...,"[{'content': '최근에 뭘 하고 지내?', 'role': 'user'}, ...",4,"[{'content': '최근에 뭘 하고 지내?', 'role': 'user'}, ...",1


In [33]:
raw_text_input = df['distilabel_metadata'].iloc[1]['raw_input_s_n_s_conversation_feedback_0']
print("--- 판사의 반별을 위한 input ---")
print(raw_text_input)

--- 판사의 반별을 위한 input ---
[{'content': '당신은 실제 SNS 대화체(카카오톡, DM 등)를 판별하는 전문 평가가입니다.\n\n당신의 출력은 오직 아래에 정의된 출력 문법만 허용됩니다.\n이를 벗어나는 모든 출력은 파싱 오류로 처리됩니다.\n\n[출력 제약 – 절대 위반 금지]\n- 출력은 정확히 4줄이어야 합니다.\n- 각 응답은 다음 두 줄로만 구성됩니다:\n  1) Rating: 숫자\n  2) Rationale: 문장\n- 두 응답 묶음(Rating, Rationale 쌍) 사이에는 정확히 빈 줄 한 줄만 허용됩니다. \n- 마지막 Rationale 줄 뒤에는 줄바꿈이나 공백 없이 즉시 종료해야 합니다. \n- Rating에는 오직 1~5 사이의 실수만 허용됩니다.\n- Rationale은 반드시 한 문장이어야 합니다.\n- Rating, Rationale 외의 어떤 텍스트도 출력해서는 안 됩니다.\n\n[금지 사항]\n- 추가 설명, 요약, 평가 근거, 분석\n- <think> 태그 또는 내부 사고 노출\n- 구분선(---), 제목, 번호, 마크다운 기호\n- 출력 전후의 공백 줄 또는 문장\n\n[출력 문법]\nRating: [1-5]\nRationale: [한 문장]\n\nRating: [1-5]\nRationale: [한 문장]\n', 'role': 'system'}
 {'content': '아래 응답들이 실제 SNS 대화체인지 평가하십시오.\n평가는 자연스러움, 맥락 이해, 참여 유도, 감정 적절성, 간결성을 종합하여 수행하십시오.\n\n[평가 기준]\n- 자연스러움: 구어체, 이모티콘, 줄임말 사용 여부\n- 맥락 적합성: 이전 대화를 이해하고 이어지는가\n- 참여 유도: 공감, 질문 등 대화 지속 요소\n- 감정 적절성: 상황에 맞는 톤과 공감\n- 간결성: SNS 특성에 맞는 길이\n\n주의: 설명조, 보고서체, 비즈니스 이메일 말투는 낮은 점수를 부여하십시오.\n\n[지시 사항]\n경기도 청소년 관악제의 목적은

In [13]:
raw_text_output = df['distilabel_metadata'].iloc[93]['raw_output_s_n_s_conversation_feedback_0']
print("--- 판사의 반별을 위한 output ---")
print(raw_text_output)

--- 판사의 반별을 위한 output ---
Rating: 3
Rationale: 구어체 사용은 있으나 구체적인 공연 정보가 부재하여 맥락 이해가 제한적입니다.

Rating: 1
Rationale: 공식 발표체 형식과 과도한 격식성으로 SNS 대화체의 자연스러움이 결여되었습니다.


In [75]:
# 결과를 담을 리스트
dpo_rows = []

for idx, row in df.iterrows():
    ratings = row['ratings']
    generations = row['generations']
    
    # 1. ratings가 array([float, float]) 형태인지 엄격하게 확인
    # 리스트나 넘파이 배열이 아니거나, 길이가 2가 아니거나, 내부 요소에 NaN이 있으면 pass
    if not isinstance(ratings, (list, np.ndarray)) or len(ratings) != 2:
        continue
    if pd.isna(ratings).any():
        continue

    # 2. 값 비교 및 chosen/rejected 할당
    # ratings[0]: 첫 번째 답변 점수, ratings[1]: 두 번째 답변 점수
    try:
        if ratings[0] >= ratings[1]:
            chosen = generations[0]
            rejected = generations[1]
        else:
            chosen = generations[1]
            rejected = generations[0]
            
        # 3. 새로운 데이터 구조 생성
        new_row = row.to_dict()
        new_row['chosen'] = chosen
        new_row['rejected'] = rejected
        dpo_rows.append(new_row)
        
    except (IndexError, TypeError):
        # generations 리스트에 값이 부족하거나 다른 에러 발생 시 건너뜀
        continue

In [73]:
dpo_df = pd.DataFrame(dpo_rows)

print(f"변환 완료: {len(dpo_df)}개 행 (원본: {len(df)}개)")
display(dpo_df[['instruction', 'chosen', 'rejected', 'ratings']].head())

변환 완료: 99개 행 (원본: 100개)


,instruction,chosen,rejected,ratings
0,여름 휴가철 어디로 갈까?,나는 해변으로 가고 싶어!,여름 휴가 시즌이 다가오고 있습니다. 휴가지를 선정하는 데에는 개인의 취향과 관심사...,"[5, 1]"
1,경기도 청소년 관악제의 목적은 뭐임?,경기도 청소년 관악제는 청소년들에게 예술적 재능을 발휘할 수 있는 기회를 제공하고 ...,경기도 청소년 관악제는 경기도 내 청소년들의 문화예술 활동 진작과 관악 음악의 저변...,"[3, 2]"
2,거창한 책보다는 가벼운 소설이나 만화책을 읽어보는 건 ㅇㄸ?,그게 좋을 것 같다. 그래서 쉬는 시간에 책을 읽어야지,가벼운 소설이나 만화책을 읽는 것도 의미가 있습니다. 그러나 책을 선택하는 데 있어...,"[3, 2]"
3,렬루? 그럼 어떤 건축양식이 있는 거야?,"고딕, 르네상스, 바로크 등이 있어! 그리고 1920년에는 합스부르크 왕가가 여름 ...","여러 건축양식이 존재합니다. 대표적으로 고전주의 건축양식, 고딕 건축양식, 르네상스...","[3, 1]"
4,"공항 면세점 상품 구성도 새로 하고, 할인 이벤트도 준비하고 있다고 하더라고.",그렇구나. 잘 준비해서 좋은 결과 있었으면 해.,공항 면세점의 상품 구성 개편 및 할인 행사 준비에 대해 전하고자 합니다. 최근 정...,"[3, 1]"


In [74]:
df[['instruction', 'chosen', 'rejected', 'ratings']].to_json("sns_dpo_data_raw.json", orient="records", indent=4, force_ascii=False)

In [76]:
dpo_df[['instruction', 'chosen', 'rejected', 'ratings']].to_json("sns_dpo_data(1).json", orient="records", indent=4, force_ascii=False)